In [1]:
# 5. How does the historical success over the last 30-40 years of the 10 most popular active directors
# influence the financial and critical success of their new movie releases?

import os
import requests
import json
from datetime import datetime
from dotenv import load_dotenv
import time
import numpy as np

load_dotenv('../../test.env')
api_key_TMDB = os.getenv("TMDB_API_KEY")
api_key_OMDB = os.getenv("OMDB_API_KEY")

if not api_key_OMDB:
    print("OMDB API KEY not found check env file")
    exit()
else:
    print("OMDB KEY loaded")

if not api_key_TMDB:
    print("TMDB API KEY not found check env file")
    exit()
else:
    print("TMDB KEY loaded")




OMDB KEY loaded
TMDB KEY loaded


In [2]:
#Building our links for the different api
tmdb_base = "https://api.themoviedb.org/3"
tmbd_person_endpoint = "/person/popular"  #Get all result for "" fitting criteria
tmdb_person_url = tmdb_base + tmbd_person_endpoint

omdb_base = f"http://www.omdbapi.com/?"

In [3]:
# Calculate 10 years ago
current_year = datetime.now().year
start_year = current_year - 10
start_date = f"{start_year}-01-01"

#Final list of the 10 most popular directors
dir_result = []
total_pages = 1

parameters = {
    'api_key' : api_key_TMDB,
    'page' : 1
}
while len(dir_result) < 10 and parameters['page'] <= total_pages:
    
    # Getting a list of people, listed on TMDb
    person_response = requests.get(tmdb_person_url, params=parameters)
    avg_director_score = 0
    newest_movie_release_year = 0000
    total_vote_count = 0
    directed_movies = []

    if person_response.status_code == 200:
        person_data = person_response.json()
        total_pages = person_data['total_pages']
        

        # Going through list of all people listed on TMDb
        for person in person_data['results']:

            # Checking if current person is Actor or Director
            if person['known_for_department'] == 'Directing':
                person_id = person['id']

                # request link to get details about each director, to check if still active and one of the most popular directors
                tmdb_credit_url = f"https://api.themoviedb.org/3/person/{person['id']}/movie_credits"
                credit_params = {
                    'api_key' : api_key_TMDB
                }
                
                try:
                    credit_response = requests.get(tmdb_credit_url, params=credit_params)
                    if credit_response.status_code == 200:
                        credit_data = credit_response.json()
                except:
                    pass

                # Filtering out niche Directors, that directed less than 5 movies
                if len(credit_data['crew'])>=5:
                    
                    # Going through movie list, that the director is credited in as crew member
                    for dir_movie in credit_data['crew']:
                        score_list=[]
                        imdb_score = 0.0
                        meta_score = 0.0
                        rotten_score = 0.0

                        # Only looking at movies, where person's job was directing 
                        if dir_movie['job'] == 'Director':


                            # Filtering out niche movies, that don't have a certain vote count and that don't have a budget listed
                            if dir_movie['vote_count']>500:

                                # Counting the amount of popular movies to calculate average score of all movies

                                movie_release_year = int(dir_movie['release_date'][0:4])

                                # Updating the newest movie release, to check, if director still active
                                if movie_release_year > newest_movie_release_year:
                                    newest_movie_release_year = movie_release_year

                                # Getting details about movie, for IMDb-ID 
                                tmdb_movie_url = f"https://api.themoviedb.org/3/movie/{dir_movie['id']}"
                                tmdb_movie_params= {
                                    'api_key' : api_key_TMDB
                                }

                                try:
                                    tmdb_movie_response = requests.get(tmdb_movie_url, params=tmdb_movie_params)
                                    if tmdb_movie_response.status_code == 200:
                                        tmdb_movie_data = tmdb_movie_response.json()
                                except:
                                    pass
                                
                                # Filtering niche movies, by only looking at bigger productions that have their budget recorded
                                if tmdb_movie_data['budget'] > 0:

                                    omdb_params = {
                                        'apikey' : api_key_OMDB,
                                        'i' : tmdb_movie_data['imdb_id'] # Using IMDb-ID, to get IMDb, Metacritic and Rotten Tomatoes score for movie
                                    }
                                    try:
                                        omdb_response = requests.get(omdb_base, params=omdb_params)
                                        if omdb_response.status_code == 200:
                                            omdb_data = omdb_response.json()
                                    except:
                                        pass
                                    
                                    # Getting a total sum of Votes on IMDb to asses the popularity of a director
                                    imdb_vote_count = omdb_data['imdbVotes']
                                    imdb_vote_count = int(imdb_vote_count.replace(',',''))
                                    total_vote_count += imdb_vote_count

                                    # Gathering TMDb, IMDb, Metacritic and Rotten Tomatoes scores and calculating their mean, to get average movie score
                                    try:
                                        tmdb_score = tmdb_movie_data['vote_average']
                                        score_list.append(tmdb_score)
                                    except:
                                        pass

                                    try:
                                        rotten_score = omdb_data['Ratings'][1]['Value']

                                        rotten_score = rotten_score[0:-1] # removing "%" from the end of the string to be able to convert string to float

                                        rotten_score = ((float(rotten_score))/10)
                                        score_list.append(rotten_score)
                                    except:
                                        pass

                                    try:
                                        meta_score = ((float(omdb_data['Metascore']))/10) # converting Metascore x/100 to decimal number x/10
                                        score_list.append(meta_score)
                                    except:
                                        pass

                                    try:
                                        imdb_score = float(omdb_data['imdbRating'])
                                        score_list.append(imdb_score)
                                    except:
                                        pass

                                    avg_movie_score = sum(score_list)/len(score_list)

                                    print(dir_movie['title'])
                                    print(person['name'])
                                    
                                    # Collecting sum of all average movie scores, to get overall score for director
                                    avg_director_score += avg_movie_score
                                    
                                    # Adding movie to the list of qualified movies to be looked at per director
                                    directed_movies.append({
                                        'title' : dir_movie['title'],
                                        'budget' : tmdb_movie_data['budget'],
                                        'revenue' : tmdb_movie_data['revenue'],
                                        'release_year' : movie_release_year,
                                        'avg_score' : round(avg_movie_score, 2),
                                        'IMDb_Vote_count' : omdb_data['imdbVotes']

                                    })

                                    # Spam proctection
                                    time.sleep(0.25)
                                
                                # Spam proctection
                                time.sleep(0.25)


                    # Filtering directors with less than 3 popular movies
                    if len(directed_movies) >= 3:

                        # Calculating average director score and log of total IMDb vote count
                        avg_director_score = avg_director_score/len(directed_movies)
                        log_votes = np.log(total_vote_count)

                        # Filtering directors by score, amount of IMDb Votes and if they're still active
                        if (
                            avg_director_score > 7
                            and newest_movie_release_year > 2019
                            and log_votes >=10.5
                        ):
                            dir_result.append((person['name'], directed_movies))

        parameters['page']+=1

        #Spam protection
        time.sleep(0.25)
                
    else:
        print(f"Error on page {parameters['page']}")
        break

print(dir_result)


Inside Out 2
Kelsey Mann
Oppenheimer
Christopher Nolan
Dunkirk
Christopher Nolan
Tenet
Christopher Nolan
Interstellar
Christopher Nolan
Inception
Christopher Nolan
The Dark Knight
Christopher Nolan
Batman Begins
Christopher Nolan
The Prestige
Christopher Nolan
Insomnia
Christopher Nolan
Memento
Christopher Nolan
Following
Christopher Nolan
The Dark Knight Rises
Christopher Nolan
Indiana Jones and the Temple of Doom
Steven Spielberg
War of the Worlds
Steven Spielberg
Raiders of the Lost Ark
Steven Spielberg
Indiana Jones and the Last Crusade
Steven Spielberg
Indiana Jones and the Kingdom of the Crystal Skull
Steven Spielberg
Minority Report
Steven Spielberg
Jurassic Park
Steven Spielberg
The Lost World: Jurassic Park
Steven Spielberg
Schindler's List
Steven Spielberg
Jaws
Steven Spielberg
The Terminal
Steven Spielberg
E.T. the Extra-Terrestrial
Steven Spielberg
Munich
Steven Spielberg
Catch Me If You Can
Steven Spielberg
A.I. Artificial Intelligence
Steven Spielberg
Close Encounters of 

In [ ]:
#list of all movies of the directors for the last 35 years
dir_movie_list = []
total_pages = 1


for person in dir_result:

    person_id=person['tmdb_id']
    director_url = f"https://api.themoviedb.org/3/person/{person_id}/movie_credits"
    parameters = {
        'api_key' : api_key_TMDB
    }

    try:
        director_response = requests.get(director_url,params=parameters)
        if director_response.status_code == 200:
            director_data = director_response.json()
    except:
        pass
    
    for movie in person['crew']:
        if movie['job'] == "Director":
            movie_id = movie['id']
            movie_url=f"https://api.themoviedb.org/3/movie/{movie_id}"
            parameters = {
                'api_key' : api_key_TMDB
            }

            try:
                movie_response = requests.get(movie_url, params=parameters)
                if movie_response.status_code == 200:
                    movie_data = movie_response.json()
            except:
                pass


            dir_movie_list.append({
                'dir_name' : person['name'],
                'dir_id' : person['tmdb_id'],
                'movie_title' : movie_data['title'],
                'movie_id' : movie_data['id'],
                'budget' : movie_data['budget'],
                'revenue' : movie_data['revenue'],
                'rating' : movie_data['vote_average'],
                'rating_count' : movie_data['vote_count'],
                'popularity' : movie_data['popularity']

            })
        
        time.sleep(0.25)

    time.sleep(0.25)

In [ ]:
for x in dir_result:
    print(x[0])
    print('hello')

Christopher Nolan
Steven Spielberg
Martin Scorsese
Quentin Tarantino
Roger Donaldson
Guy Ritchie
Tim Burton
Taika Waititi
Gurinder Chadha
Alejandro González Iñárritu
